In [15]:
from langchain.chat_models import init_chat_model
from langgraph.graph import StateGraph, START, END
from langchain_core.prompts import PromptTemplate
from typing import TypedDict
import os
from dotenv import load_dotenv
load_dotenv()

True

In [16]:
os.environ["AZURE_OPENAI_API_KEY"] = os.getenv("AZURE_OPENAI_API_KEY")
os.environ["AZURE_OPENAI_ENDPOINT"] = os.getenv("AZURE_OPENAI_ENDPOINT")
os.environ["OPENAI_API_VERSION"] = os.getenv("OPENAI_GPT_API_VERSION")

model = init_chat_model("azure_openai:gpt-5-mini")


In [17]:
class LLMState(TypedDict):
    question: str
    answer: str

graph = StateGraph(LLMState)

def llm_qa(state: LLMState) -> LLMState:
    question = state['question']

    template = PromptTemplate(
        template="Answer the following question:\n {question}",
        input_variables=['question']
    )

    prompt = template.invoke({'question': question})

    response = model.invoke(prompt).content

    state['answer'] = response
    return state

graph.add_node('llm_qa', llm_qa)

In [18]:
graph.add_edge(START, 'llm_qa')
graph.add_edge('llm_qa', END)

workflow = graph.compile()

In [19]:
#exxecute

initial_state = {
    'question': "How far is moon from the earth?"
}

final_state = workflow.invoke(initial_state)
print(final_state['answer'])

The Moon’s average distance from Earth is about 384,400 km (238,855 miles) measured center-to-center.

Because the Moon’s orbit is elliptical, the distance varies:
- Perigee (closest): about 363,300 km (225,623 miles)
- Apogee (farthest): about 405,500 km (251,966 miles)

Light takes roughly 1.28 seconds to travel one way between Earth and the Moon.
